# Contextual Bandit 模型表现可视化

这个 notebook 用来可视化项目里的推荐策略和 OPE 估计结果。它默认读取：

- `experiment_results.csv`: 核心实验结果
- `policy_sweep_results.csv`: 参数 sweep 结果，如果存在就展示

建议先运行：

```bash
python run_experiments.py --bootstrap-samples 500
python run_policy_sweep.py --bootstrap-samples 100
```

如果当前环境没有安装 `obp`，可以先使用已有的 `experiment_results.csv` 查看基础图表。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

PROJECT_ROOT = Path.cwd()
CORE_PATH = PROJECT_ROOT / "experiment_results.csv"
SWEEP_PATH = PROJECT_ROOT / "policy_sweep_results.csv"

ESTIMATORS = ["ips", "snips", "dm", "dr"]

## 1. 读取实验结果

In [ ]:
if not CORE_PATH.exists():
    raise FileNotFoundError(
        "Missing experiment_results.csv. Run `python run_experiments.py --bootstrap-samples 500` first."
    )

core = pd.read_csv(CORE_PATH)
sweep = pd.read_csv(SWEEP_PATH) if SWEEP_PATH.exists() else pd.DataFrame()

available_estimators = [col for col in ESTIMATORS if col in core.columns]
behavior_value = float(core["behavior_mean_reward"].iloc[0])

print(f"Loaded core results: {CORE_PATH.name}, shape={core.shape}")
if sweep.empty:
    print("No policy_sweep_results.csv found yet. Sweep visualizations will be skipped.")
else:
    print(f"Loaded sweep results: {SWEEP_PATH.name}, shape={sweep.shape}")

display(core)

## 2. 各策略的 OPE 估计值

这里比较 IPS、SNIPS、DM、DR 对每个 target policy 的 policy value 估计。虚线是 logged behavior policy 的平均 reward。

In [ ]:
n_estimators = len(available_estimators)
n_cols = 2
n_rows = int(np.ceil(n_estimators / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 4.8 * n_rows), squeeze=False)

for ax, estimator in zip(axes.ravel(), available_estimators):
    plot_df = core.sort_values(estimator, ascending=True).reset_index(drop=True)
    y = np.arange(len(plot_df))
    values = plot_df[estimator].to_numpy()

    ax.barh(y, values, color=sns.color_palette("deep", len(plot_df)))
    if f"{estimator}_ci_lower" in plot_df.columns and f"{estimator}_ci_upper" in plot_df.columns:
        lower = plot_df[f"{estimator}_ci_lower"].to_numpy()
        upper = plot_df[f"{estimator}_ci_upper"].to_numpy()
        xerr = np.vstack([values - lower, upper - values])
        ax.errorbar(values, y, xerr=xerr, fmt="none", color="black", capsize=3, linewidth=1)

    ax.axvline(behavior_value, color="black", linestyle="--", linewidth=1.2, label="behavior mean reward")
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["policy"])
    ax.set_xlabel("estimated policy value")
    ax.set_title(estimator.upper())
    ax.legend(loc="lower right")

for ax in axes.ravel()[n_estimators:]:
    ax.axis("off")

fig.suptitle("OPE estimates by target policy", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

## 3. 相对 behavior policy 的提升倍数

大于 1 表示 OPE 估计该策略优于历史行为策略；小于 1 表示低于历史行为策略。

In [ ]:
relative = core.melt(
    id_vars=["policy"],
    value_vars=available_estimators,
    var_name="estimator",
    value_name="estimated_value",
)
relative["relative_to_behavior"] = relative["estimated_value"] / behavior_value

plt.figure(figsize=(13, 6))
sns.barplot(data=relative, x="relative_to_behavior", y="policy", hue="estimator", orient="h")
plt.axvline(1.0, color="black", linestyle="--", linewidth=1.2)
plt.xlabel("estimated value / behavior mean reward")
plt.ylabel("")
plt.title("Estimated improvement over the logged behavior policy")
plt.legend(title="estimator", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. OPE 稳定性诊断

确定性策略如果和 logged action 重合率很低，IPS/DR 往往方差很高。这里用 `logged_action_match_rate`、`mean_importance_weight` 和 optional `effective_sample_size_ratio` 看估计是否可信。

In [ ]:
diagnostic = core.copy()
if "effective_sample_size_ratio" not in diagnostic.columns:
    diagnostic["effective_sample_size_ratio"] = np.nan

metric_for_y = "dr" if "dr" in diagnostic.columns else available_estimators[0]
sizes = diagnostic["effective_sample_size_ratio"].fillna(0.2).clip(lower=0.02) * 2500

plt.figure(figsize=(11, 6))
plt.scatter(
    diagnostic["logged_action_match_rate"],
    diagnostic[metric_for_y],
    s=sizes,
    c=diagnostic["mean_importance_weight"],
    cmap="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.6,
)
for _, row in diagnostic.iterrows():
    plt.annotate(row["policy"], (row["logged_action_match_rate"], row[metric_for_y]), xytext=(5, 4), textcoords="offset points", fontsize=9)

plt.axhline(behavior_value, color="black", linestyle="--", linewidth=1.1)
plt.colorbar(label="mean importance weight")
plt.xlabel("logged action match rate")
plt.ylabel(f"{metric_for_y.upper()} estimated policy value")
plt.title("Estimator reliability diagnostics")
plt.tight_layout()
plt.show()

cols = ["policy", "logged_action_match_rate", "mean_importance_weight"]
if "effective_sample_size" in core.columns:
    cols += ["effective_sample_size", "effective_sample_size_ratio"]
display(core[cols].sort_values("logged_action_match_rate"))

## 5. 参数 sweep：epsilon、alpha、sample scale

如果已经运行 `run_policy_sweep.py`，这里会展示不同超参数下的估计表现。

In [ ]:
if sweep.empty:
    print("Skip: policy_sweep_results.csv not found. Run `python run_policy_sweep.py --bootstrap-samples 100` to enable this section.")
else:
    sweep_estimators = [col for col in ESTIMATORS if col in sweep.columns]
    sweep_long = sweep.melt(
        id_vars=["policy_family", "sweep_parameter", "sweep_value", "policy"],
        value_vars=sweep_estimators,
        var_name="estimator",
        value_name="estimated_value",
    )
    sweep_long = sweep_long[sweep_long["sweep_parameter"] != "none"]

    families = sweep_long["policy_family"].dropna().unique()
    fig, axes = plt.subplots(len(families), 1, figsize=(12, 4.5 * len(families)), squeeze=False)
    for ax, family in zip(axes.ravel(), families):
        family_df = sweep_long[sweep_long["policy_family"] == family]
        sns.lineplot(
            data=family_df,
            x="sweep_value",
            y="estimated_value",
            hue="estimator",
            marker="o",
            ax=ax,
        )
        ax.axhline(behavior_value, color="black", linestyle="--", linewidth=1.1)
        parameter = family_df["sweep_parameter"].iloc[0]
        ax.set_title(f"{family}: OPE value across {parameter}")
        ax.set_xlabel(parameter)
        ax.set_ylabel("estimated policy value")
        ax.legend(title="estimator", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

    best_by_dr = sweep.sort_values("dr", ascending=False) if "dr" in sweep.columns else sweep
    display(best_by_dr.head(10))

## 6. OBP 官方估计器对照

如果结果里有 `obp_*_abs_diff` 列，说明脚本成功调用了 OBP 官方 `OffPolicyEvaluation`。这里检查手写 estimator 和官方 estimator 的绝对差。

In [ ]:
diff_cols = [col for col in core.columns if col.startswith("obp_") and col.endswith("_abs_diff")]

if not diff_cols:
    status = core["obp_status"].iloc[0] if "obp_status" in core.columns else "not available in this CSV"
    print(f"Skip: no OBP parity columns found. obp_status={status}")
else:
    parity = core.melt(
        id_vars=["policy"],
        value_vars=diff_cols,
        var_name="estimator",
        value_name="absolute_diff",
    )
    parity["estimator"] = parity["estimator"].str.replace("obp_", "", regex=False).str.replace("_abs_diff", "", regex=False)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=parity, x="absolute_diff", y="policy", hue="estimator", orient="h")
    plt.xlabel("absolute difference from OBP official estimate")
    plt.ylabel("")
    plt.title("Manual estimator vs OBP parity check")
    plt.legend(title="OBP estimator", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    display(parity.sort_values("absolute_diff", ascending=False))

## 7. 自动总结表

默认用 DR 排名，因为 DR 同时利用 propensity score 和 reward model；如果 reward model 很弱或 action overlap 很差，需要结合上一节诊断谨慎解释。

In [ ]:
ranking_metric = "dr" if "dr" in core.columns else available_estimators[0]
summary_cols = ["policy", ranking_metric, "behavior_mean_reward", "logged_action_match_rate", "mean_importance_weight"]
if "effective_sample_size_ratio" in core.columns:
    summary_cols.append("effective_sample_size_ratio")

summary = core[summary_cols].copy()
summary["relative_to_behavior"] = summary[ranking_metric] / summary["behavior_mean_reward"]
summary = summary.sort_values(ranking_metric, ascending=False).reset_index(drop=True)
display(summary)

best = summary.iloc[0]
print(
    f"Best by {ranking_metric.upper()}: {best['policy']} "
    f"({ranking_metric}={best[ranking_metric]:.6f}, "
    f"relative={best['relative_to_behavior']:.2f}x behavior)."
)

low_overlap = summary[summary["logged_action_match_rate"] < 0.05]
if not low_overlap.empty:
    print("Warning: some top policies may have low logged-action overlap; inspect IPS/DR uncertainty before making strong claims.")